# 02 Epi Visualization: 5 Key Charts

Using the Pine Cedar Nursing Home Legionnaires' disease line list, learn the three major plotting packages: matplotlib / seaborn / plotly.

| Chart | Package | What to look for |
|------|---------|----------|
| Epidemic curve | matplotlib | Transmission mode (common source vs ongoing transmission) |
| Age distribution | seaborn | Whether age is a risk factor |
| Attack rate by wing | seaborn | Clues to spatial clustering |
| Severity × comorbidity | seaborn heatmap | Multi-factor interaction |
| Interactive stratified curve | plotly | Comparing epidemic peaks across floors |

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Data preparation (same as the previous lesson) ---
# This cell does three things:
#   1) import the plotting packages (matplotlib, seaborn, plotly)
#   2) set global chart style (fonts, resolution)
#   3) read the data and build derived variables
#
# 💡 The roles of the three plotting packages:
#   matplotlib → the low-level engine, like laying bricks to build a house yourself (maximum flexibility)
#   seaborn    → a high-level wrapper over matplotlib, like buying a prefab home (write less, do more; great for statistical charts)
#   plotly     → an interactive charting engine, hover/zoom with the mouse (good for presentations and the web)

import pathlib

import pandas as pd
import matplotlib.pyplot as plt       # plt = the conventional abbreviation for matplotlib
import matplotlib.font_manager as fm  # fm = the font manager (handles Chinese fonts)
import seaborn as sns                 # sns = the conventional abbreviation for seaborn
import plotly.express as px           # px = plotly's quick plotting interface
import plotly.io as pio               # pio = plotly's input/output settings

# -- Global chart style settings --
# plt.style.use("ggplot") → apply the ggplot style (light gray background + white gridlines, common in academic papers)
# plt.rcParams["figure.dpi"] = 150 → increase resolution (the default 100 is too blurry on screen)
# 💡 rcParams = "runtime configuration parameters", controlling all of matplotlib's defaults
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# -- CJK font setup (avoid Chinese labels showing as boxes □□□) --
# Problem: by default matplotlib only recognizes Latin fonts; Chinese characters turn into "tofu blocks"
# Solution: manually scan the system font directories and register all CJK (Chinese/Japanese/Korean) fonts
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

# Set the Chinese font candidate list (tried in priority order)
plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False  # prevent the minus sign from showing as a box

# Plotly: make sure interactive charts still render during a static build (jupyter-book build)
pio.renderers.default = "notebook"

# --- Read the data ---
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

# Date conversion (same as Step 3)
date_cols = [
    "facility_admission_date", "symptom_onset_date",
    "hospitalization_date", "death_date", "notification_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Derived variables (same as Step 4)
comorbidity_cols = [
    "comorbidity_chf", "comorbidity_dm",
    "comorbidity_cancer", "comorbidity_copd", "immunosuppressed",
]
df["n_comorbidities"] = df[comorbidity_cols].sum(axis=1)
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["age_group"] = pd.cut(
    df["age"], bins=[59, 69, 79, 89, 100],
    labels=["60-69", "70-79", "80-89", "90+"],
)

# Take only the infected (for the epidemic curve, etc.)
# df[df["infected"] == 1] → boolean filter, keep only rows with infected=1
# .copy() → make an independent copy to avoid a SettingWithCopyWarning when modifying
cases = df[df["infected"] == 1].copy()
print(f"Total: {len(df)} people, infected: {len(cases)} people")

## 1) Epidemic Curve (matplotlib)

The most iconic epi chart. The X-axis is the date of symptom onset and the Y-axis is the number of new cases per day.
The shape of the curve lets you infer the transmission mode: a sharp peak → common source; a long tail → ongoing transmission.

### Key drawing points (CDC / ECDC standards)

An epidemic curve is essentially a **histogram**, not an ordinary bar chart:

- **No gaps between adjacent bars**: the X-axis is a continuous time axis, so there should be no space between bars (`width=1.0`)
- **Fill in dates with no cases**: even a day with 0 cases must hold its place (fill with `reindex`), otherwise the X-axis spacing is distorted
- **Show the pre-outbreak background period**: include dates 1–2 incubation periods before the outbreak so readers can see when it deviated from baseline
- **The title should be self-contained**: include the disease name, location, and time range
- **X-axis**: label it "Date of Symptom Onset"—clearly state the time basis
- **Y-axis**: label it "Number of Cases"—must be integer ticks, starting at 0, not truncated
- **Hide gridlines**: reduce visual clutter, remove the top and right spines
- **Distinguish case classification by color**: confirmed vs probable must use different colors with a legend
- **Don't label numbers on the bars**: avoid interference between digital and analog information

In [ ]:
# --- Drawing the epidemic curve ---
# The core pattern of matplotlib plotting: fig, ax = plt.subplots()
#   fig = figure (the whole sheet of paper)
#   ax  = axes (the drawing area on the paper)
#   All plotting commands operate on ax (ax.bar, ax.set_title, ...)
#
# 💡 Why not use plt.plot()?
#   plt.plot() is "easy mode", fine for a single chart
#   fig, ax is "professional mode", letting you place multiple subplots on one sheet
#   Academic papers and investigation reports almost always use the fig, ax pattern

import matplotlib.dates as mdates  # date formatting tools

# 1) Compute daily case counts
daily = cases.groupby("symptom_onset_date").size().rename("cases")

# 2) Fill in the full date range (including 3 days before the outbreak, to show the background period)
#    pd.date_range() → produce a continuous sequence of dates
#    .reindex(fill_value=0) → fill dates with no cases with 0 (don't leave them blank!)
date_range = pd.date_range(
    daily.index.min() - pd.Timedelta(days=3),
    daily.index.max() + pd.Timedelta(days=1),
    freq="D",
)
daily = daily.reindex(date_range, fill_value=0)

# 3) Draw the bar chart
fig, ax = plt.subplots(figsize=(10, 4))  # figsize=(width, height) in inches
ax.bar(
    daily.index, daily.values,
    width=1.0,                         # width=1 (day), bars sit flush together
    color="#2c7fb8",                    # bar fill color
    edgecolor="white", linewidth=0.5,  # white edges make the bars distinguishable
)

# 4) Title and axis labels
# 💡 A good title = disease + location + time, so the image makes sense on its own
ax.set_title(
    "Pine Cedar Nursing Home Legionnaires' Disease Epidemic Curve, by Date of Symptom Onset, January 2026",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("Date of Symptom Onset")
ax.set_ylabel("Number of Cases")

# 5) Date formatting
# DateFormatter("%m/%d") → show "month/day" format
# DayLocator(interval=2) → put a tick every 2 days
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45)  # rotate date labels 45 degrees to avoid overlap

# 6) Fine-tune the axes
ax.set_xlim(
    daily.index.min() - pd.Timedelta(hours=12),
    daily.index.max() + pd.Timedelta(hours=12),
)
ax.set_ylim(bottom=0)                              # Y-axis starts at 0
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))  # Y-axis shows only integers

# 7) CDC style: hide gridlines, remove top and right spines (for a cleaner chart)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()  # auto-adjust margins so labels aren't clipped
plt.show()

### The classic unit-chart epidemic curve

The unit chart (stacked squares) epidemic curve common in textbooks and CDC investigation reports—each little square represents one case, stacked into a column. Here we use color to distinguish confirmed and probable cases.

> 💡 The unit chart is especially suited to **small clusters** (a few dozen to a hundred-something cases). When there are too many cases the squares become too small, and a standard histogram is more appropriate.

In [ ]:
# --- Unit-chart epidemic curve ---
# Each little square = one case, colored to distinguish confirmed vs probable
# Suited to small clusters (a few dozen to a hundred-something); use a standard histogram for large outbreaks
#
# Technical points:
#   plt.Rectangle(xy, width, height) → draw a rectangle
#   ax.add_patch(rect) → add the rectangle to the plot
#   mdates.date2num(date) → convert a date to matplotlib's numeric coordinate

# Prepare the daily confirmed / probable case counts
# .unstack(fill_value=0) → move case_classification from "rows" to "columns"
daily_class = (
    cases.groupby(["symptom_onset_date", "case_classification"])
    .size()
    .unstack(fill_value=0)
)
daily_class = daily_class.reindex(date_range, fill_value=0)
colors_map = {"confirmed": "#2c7fb8", "probable": "#a6bddb"}

fig, ax = plt.subplots(figsize=(10, 5))
box_size = 1.0

for date in daily_class.index:
    x = mdates.date2num(date)      # date → numeric coordinate
    j = 0                          # j = current stacking height (start at 0 and stack upward)
    for cls in ["confirmed", "probable"]:
        count = daily_class.at[date, cls] if cls in daily_class.columns else 0
        for _ in range(int(count)):  # draw one square per case
            rect = plt.Rectangle(
                (x - box_size / 2, j * box_size),  # bottom-left corner
                box_size, box_size,                  # width, height
                facecolor=colors_map[cls],
                edgecolor="white", linewidth=0.8,
            )
            ax.add_patch(rect)
            j += 1

# Axis settings
ax.set_xlim(
    mdates.date2num(daily_class.index.min()) - 1.5,
    mdates.date2num(daily_class.index.max()) + 1.5,
)
y_max = daily_class.sum(axis=1).max()
ax.set_ylim(0, y_max + 1)
ax.set_aspect("equal")  # make the squares actually square (width = height)

ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

ax.set_title(
    "Pine Cedar Nursing Home Legionnaires' Disease Epidemic Curve — Unit Chart (by Case Classification)",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("Date of Symptom Onset")
ax.set_ylabel("Number of Cases")

# Manual legend (because we drew with add_patch, matplotlib won't generate a legend automatically)
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#2c7fb8", edgecolor="white", label="Confirmed"),
    Patch(facecolor="#a6bddb", edgecolor="white", label="Probable"),
]
ax.legend(handles=legend_elements, loc="upper left", frameon=False)

ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 2) Age Distribution: Infected vs. Not Infected (seaborn)

Overlay the age distribution of all 280 people to see whether the infected are concentrated in a specific age band.

In [ ]:
# --- Age distribution chart (seaborn) ---
# seaborn is a high-level wrapper over matplotlib; one line draws an attractive statistical chart
#
# sns.histplot() parameters:
#   data=df        → the data source (the whole DataFrame)
#   x="age"        → use the age column for the X-axis
#   hue="infected" → color by the infected column (0=not infected, 1=infected)
#   hue_order=[1,0]→ legend order: show the infected first
#   bins=15        → split into 15 bins (too few hides detail, too many is fragmented)
#   multiple="stack" → stacked mode (infected stacked on top of not-infected)
#   palette={1:"red", 0:"gray"} → custom color mapping
#   ax=ax          → which axes to draw on
#
# 💡 The difference between seaborn and matplotlib:
#   matplotlib: ax.bar(x, y, color=...) — you prepare the x, y data yourself
#   seaborn: sns.histplot(data=df, x="age") — just hand it the DataFrame and it computes for you

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(
    data=df, x="age", hue="infected", hue_order=[1, 0], bins=15,
    multiple="stack", palette={1: "#e34a33", 0: "#cccccc"}, ax=ax,
)
ax.set_title("Age Distribution: Infected vs. Not Infected")
ax.set_xlabel("Age")
ax.set_ylabel("Count")
ax.legend(title="Infection", labels=["Infected", "Not infected"])
plt.tight_layout()
plt.show()

## 3) Attack Rate by Wing Bar Chart (seaborn)

You can't compare case counts directly—you have to divide by the denominator (number of residents) to be fair.
A wing with an unusually high attack rate may have a common exposure source (e.g. shower equipment).

In [ ]:
# --- Attack rate by wing bar chart (seaborn) ---
# sns.barplot() parameters:
#   data=wing_stats → the data source
#   x="label"       → use the wing label for the X-axis (e.g. "1A", "2B")
#   y="attack_rate_pct" → use the attack-rate percentage for the Y-axis
#   hue="label"     → a different color per wing
#   palette="YlOrRd"→ yellow→orange→red gradient palette (higher = redder)
#
# 💡 Why use the attack rate rather than the case count?
#   Wing A: 30 of 50 infected (60%) vs Wing B: 30 of 100 infected (30%)
#   Same case count (both 30), but Wing A's risk is twice as high!
#   The denominator matters → always divide by the denominator

# Compute wing statistics
wing_stats = (
    df.groupby(["floor", "wing"])
    .agg(residents=("case_id", "size"), infected=("infected", "sum"))
    .reset_index()
)
wing_stats["attack_rate_pct"] = (
    wing_stats["infected"] / wing_stats["residents"] * 100
).round(1)
wing_stats["label"] = wing_stats["floor"].astype(str) + wing_stats["wing"]
wing_stats = wing_stats.sort_values("attack_rate_pct", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(
    data=wing_stats, x="label", y="attack_rate_pct",
    hue="label", palette="YlOrRd", legend=False, ax=ax,
)
ax.set_title("Attack Rate by Wing")
ax.set_xlabel("Wing")
ax.set_ylabel("Attack Rate (%)")

# Label numbers above the bars — so readers can read the exact value without checking the Y-axis
# itertuples() → turn each row into a tuple, faster than iterrows()
for i, row in enumerate(wing_stats.itertuples()):
    ax.text(i, row.attack_rate_pct + 1, f"{row.attack_rate_pct}%",
            ha="center", fontsize=10)

plt.tight_layout()
plt.show()

## 4) Severity × Comorbidity Count Heatmap (seaborn)

Are people with more comorbidities more prone to severe disease? Cross-compare with a heatmap.

In [ ]:
# --- Severity × comorbidity count heatmap (seaborn) ---
# A heatmap = a 2D table where color intensity represents magnitude
# Good for observing "the cross-relationship between two categorical variables"
#
# sns.heatmap() parameters:
#   heat_data     → a 2D table (rows=severity, columns=comorbidity count)
#   annot=True    → show the number in each cell
#   fmt="d"       → number format as integer (d=digit)
#   cmap="YlOrRd" → palette: yellow→orange→red (larger = redder)
#
# Data preparation flow:
#   1) filter symptomatic infected people (exclude not_ill and asymptomatic)
#   2) groupby the two columns → .size() counts people per group
#   3) .unstack() moves comorbidity count from rows to columns (into a 2D table)
#   4) .reindex() ensures severity is ordered mild → moderate → severe

severity_order = ["mild", "moderate", "severe"]
heat_data = (
    cases[cases["clinical_severity"].isin(severity_order)]
    .groupby(["clinical_severity", "n_comorbidities"])
    .size()
    .unstack(fill_value=0)
    .reindex(severity_order)  # ensure the row order
)

fig, ax = plt.subplots(figsize=(8, 3.5))
sns.heatmap(heat_data, annot=True, fmt="d", cmap="YlOrRd", ax=ax)
ax.set_title("Clinical Severity × Number of Comorbidities")
ax.set_xlabel("Number of Comorbidities")
ax.set_ylabel("Severity")
plt.tight_layout()
plt.show()

## 5) Interactive Stratified Epidemic Curve (Plotly)

Use Plotly to color the epidemic curve by floor, with values on hover. Plotly's interactive charts must follow the same CDC epidemic-curve standards: no gaps (`bargap=0`), a descriptive title, hidden gridlines, and a Y-axis starting at 0.

Observe: are the epidemic peaks of the three floors synchronized? If not, what does that mean?

In [ ]:
# --- Interactive stratified epidemic curve (Plotly) ---
# Plotly's strengths: values on hover, zoomable, exportable as HTML
#
# px.bar() parameters (differences from matplotlib):
#   x="column_name" → give the column name directly (no need to precompute)
#   y="column_name" → same
#   color="floor"  → color by the floor column (auto-generates a legend)
#   barmode="stack" → stacked mode
#   color_discrete_sequence=[...] → custom color sequence
#   title="..."    → title (one parameter does it)
#   labels={...}   → custom axis labels (a dict mapping)
#
# 💡 The core difference between Plotly and matplotlib:
#   matplotlib: "imperative" — tell it how to draw step by step (set_title, set_xlabel...)
#   plotly:     "declarative" — tell it what you want and it draws (one function does it)

import plotly.express as px
import plotly.graph_objects as go

# Stratify by floor and fill in the full date range
daily_floor = (
    cases.groupby(["symptom_onset_date", "floor"])
    .size()
    .rename("cases")
    .reset_index()
)
daily_floor["floor"] = daily_floor["floor"].astype(str) + "F"

# Fill in all date × floor combinations (including days with 0 cases)
all_dates = pd.date_range(
    cases["symptom_onset_date"].min() - pd.Timedelta(days=3),
    cases["symptom_onset_date"].max() + pd.Timedelta(days=1),
    freq="D",
)
all_floors = sorted(daily_floor["floor"].unique())
full_idx = pd.MultiIndex.from_product(
    [all_dates, all_floors], names=["symptom_onset_date", "floor"]
)
daily_floor = (
    daily_floor.set_index(["symptom_onset_date", "floor"])
    .reindex(full_idx, fill_value=0)
    .reset_index()
)

fig = px.bar(
    daily_floor,
    x="symptom_onset_date", y="cases", color="floor",
    barmode="stack",
    color_discrete_sequence=["#2c7fb8", "#41ae76", "#fe9929"],
    title="Pine Cedar Nursing Home Legionnaires' Disease Epidemic Curve, by Floor and Date of Symptom Onset, January 2026",
    labels={"symptom_onset_date": "Date of Symptom Onset",
            "cases": "Number of Cases",
            "floor": "Floor"},
)

# fig.update_layout() → Plotly's "fine-tuning" method (like matplotlib's ax.set_xxx)
fig.update_layout(
    bargap=0,                              # no gaps between bars (CDC standard)
    xaxis=dict(showgrid=False),            # hide vertical gridlines
    yaxis=dict(showgrid=False, rangemode="tozero"),  # Y-axis starts at 0
    plot_bgcolor="white",                  # white background
    # place the legend above the chart (horizontally)
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig.show()

## 6) Exporting Charts—Investigation Reports and Journal Submission

How do you save a finished chart? Different uses have different format and resolution requirements:

| Use | Format | DPI | Notes |
|------|------|-----|------|
| Investigation report / presentation | PNG | 150–300 | Raster, small file, good for Word/PPT |
| Journal submission | PDF / SVG | Vector | Enlarges infinitely without loss, top choice for typesetting |
| Web / interactive | HTML | — | Plotly only, keeps interactivity |

### Journal submission specs at a glance

| Journal | Resolution | Single-column width | Font | Special requirement |
|------|--------|---------|------|---------|
| NEJM | ≥1000 DPI | 8.9 cm | Arial | Colorblind-safe |
| Lancet | ≥300 DPI | 8.5 cm | Arial | Legible in pure black and white |
| JAMA | ≥350 DPI | 8.4 cm | Arial | EPS or PDF |

In [ ]:
# --- Chart export examples ---

# ===== matplotlib export =====
# fig.savefig() parameters:
#   "name.png"         → output file path (the extension decides the format: .png/.pdf/.svg)
#   dpi=300            → resolution (300 for investigation reports, ≥600 for journals)
#   bbox_inches="tight"→ auto-crop excess whitespace (very important! without it you get a big white margin)
#   facecolor="white"  → set the background to white (the default may be transparent)

# Redraw the epidemic curve as an export demo
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily.index, daily.values, width=1.0,
       color="#2c7fb8", edgecolor="white", linewidth=0.5)
ax.set_title("Pine Cedar Nursing Home Legionnaires' Disease Epidemic Curve, by Date of Symptom Onset, January 2026",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Date of Symptom Onset")
ax.set_ylabel("Number of Cases")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45)
ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Export as PNG (for investigation reports)
fig.savefig("epi_curve_report.png", dpi=300, bbox_inches="tight", facecolor="white")

# Export as PDF (for journal submission, vector image with no loss)
fig.savefig("epi_curve_journal.pdf", bbox_inches="tight", facecolor="white")

print("✅ Exported: epi_curve_report.png (300 DPI) + epi_curve_journal.pdf (vector image)")
plt.show()

# ===== Plotly export =====
# fig.write_html("name.html")  → keeps interactivity, good for web reports
# fig.write_image("name.png")  → static image (requires the kaleido package)
# 💡 Tip: write_image needs `uv add kaleido`

## Summary

In this lesson you learned 5 common epi charts + chart export:

| Chart | Package | What to look for |
|------|------|----------|
| Epidemic curve | matplotlib | Peak timing, rate of rise/fall → transmission mode |
| Age distribution | seaborn | Whether the infected are concentrated in a specific age band |
| Wing bar chart | seaborn | Which wings have an unusually high attack rate → spatial clues |
| Severity × comorbidity | seaborn heatmap | Whether people with more comorbidities are more prone to severe disease |
| Interactive stratified curve | plotly | Whether the epidemic peaks are synchronized across floors |

### The three plotting packages at a glance

| Package | Best for | Core syntax |
|------|---------|---------|
| **matplotlib** | Full customization, journal submission | `fig, ax = plt.subplots()` → `ax.bar()` → `ax.set_title()` |
| **seaborn** | Statistical charts (histograms, heatmaps) | `sns.histplot(data=df, x="age", hue="infected")` |
| **plotly** | Interactive charts, web reports | `px.bar(df, x="date", y="cases", color="floor")` |

### Chart export at a glance

| Method | Format | Use |
|------|------|------|
| `fig.savefig("chart.png", dpi=300, bbox_inches="tight")` | PNG | Investigation report / presentation |
| `fig.savefig("chart.pdf", bbox_inches="tight")` | PDF | Journal submission (vector image) |
| `fig.write_html("chart.html")` | HTML | Interactive web report (Plotly) |

In the next chapter (Ch03 Descriptive Statistics), we'll quantify these observations—computing 2×2 tables, chi-square tests, and risk ratios.